# Does right heart catheterization harm critically ill patients, and for whom?
### A causal machine learning analysis of the SUPPORT data (Connors et al., 1996) along the 8-step pipeline

**Research question.** What is the causal effect of right heart catheterization (RHC) in the first 24 hours of intensive care on 30-day mortality, how strong would unmeasured confounding have to be to explain the gap to the randomized evidence, and does the effect differ between patients?

**Notation.** Treatment $W_i$ = `swang1` (1 = RHC on day 1), outcome $Y_i$ = `dth30` (1 = died within 30 days), covariates $X_i$ = patient characteristics recorded on day 1, before or at the time of the treatment decision. Potential outcomes $Y_i(1), Y_i(0)$. ATE $\tau = E[Y_i(1) - Y_i(0)]$, a risk difference reported in percentage points. CATE $\tau(x) = E[Y_i(1) - Y_i(0) \mid X_i = x]$. Nuisances $\hat m(x) \approx E[Y \mid X = x]$ and $\hat e(x) \approx P[W = 1 \mid X = x]$.

**Pipeline** (eight steps, two feedback loops):

| Step | What happens here | Main output |
|---|---|---|
| 1. Research question and data collection | load data, check missing covariates | analysis sample |
| 2. Causal structure | DAG with day-1 confounders, backdoor check | DAG figure |
| 3. Causal estimand | ATE (primary), overlap population, CATE (secondary) | estimand definition |
| 4. Identifying assumptions | SUTVA, overlap, balance | overlap histogram, love plot |
| 5. Fit the method | nuisances, causal forest, estimator ladder | ATE with CI vs. published estimates |
| 6. Evaluate | calibration test, heterogeneity check, RATE | is the heterogeneity real? |
| 7. Robustness | placebo, random cause, semi-synthetic, sensitivity benchmarked against the RCTs | robustness scorecard |
| 8. Interpretation | best linear projection, partial dependence, summary | numbers for the story |

Steps 1 to 4 **identify** (settled before any model is fitted), steps 5 to 8 **estimate and interrogate**.

**How to run.** Always use *Restart Kernel* and then *Run All*. Running cells out of order leaves stale variables in memory and the results can be silently wrong. The first run downloads the data (2.3 MB) into `data/`.

> **Status: work in progress.** Setup and Steps 1.1 to 1.2 are in place. The remaining steps follow.

## 0. Setup
Packages: `econml` (generalized random forests, the Python counterpart of R's `grf`), `statsmodels`, `scikit-learn`, `networkx`, `matplotlib`. The data are downloaded from the Vanderbilt Biostatistics data repository and checked against a SHA-256 checksum, so every run uses exactly the same file.

In [1]:
# If something is missing, install it once from a NOTEBOOK cell (not from the terminal):
# %pip install econml statsmodels networkx
# If pip wants to downgrade numpy or scikit-learn in your base environment, use a fresh
#    environment instead:  conda create -n causal python=3.11  (then install the packages there)

import hashlib
import json
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import networkx as nx
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

from econml.grf import CausalForest, RegressionForest

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_rows", 100)   # show the full data dictionary

# ------------------------------------------------------------------ configuration
SEED = 42                    # one seed for everything: every number below is reproducible
N_TREES = 2000               # forests in the main analysis (econml needs a multiple of 4 for inference)
N_TREES_REFUTE = 1000        # lighter forests for the many refits in Step 7
LEAF_GRID = [5, 10, 20, 40]  # candidate min_samples_leaf values for the causal forest (revise loop)
N_FOLDS = 5                  # cross-fitting folds for the parametric nuisance models
ALPHA = 0.05
Z_CRIT = stats.norm.ppf(1 - ALPHA / 2)

# Published references for validation (Step 5) and for the sensitivity benchmark (Step 7)
REF = {
    "connors_or": {"estimate": 1.24, "ci": (1.03, 1.49),
                   "source": "Connors et al. (1996), JAMA 276:889-897, propensity-matched OR for 30-day mortality"},
    "hirano_imbens_rd": {"range": (0.053, 0.062),
                         "source": "Hirano & Imbens (2001), Health Serv Outcomes Res Methodol 2:259-278, "
                                   "weighting estimates of the increase in 30-day mortality"},
    "rct_meta_or": {"estimate": 1.04, "ci": (0.90, 1.20),
                    "source": "Shah et al. (2005), JAMA 294:1664-1670, meta-analysis of 13 RCTs (5,051 patients)"},
}

rng = np.random.default_rng(SEED)
# Outputs go to the repository root, whether the notebook runs from notebooks/ or from the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)
DATA_DIR = ROOT / "data"
RESULTS = {}                 # registry of key numbers, exported at the end for the background document

# ------------------------------------------------------------------ plot style
# Data Visualization Principles: declutter (no top/right spines, no tick marks, no grid),
#    one accent colour as the only preattentive highlight, direct labels instead of legends,
#    keep the uncertainty and a source line ("expose the lineage").
GREY, LIGHT, DARK = "#9A9A9A", "#D9D9D9", "#2B2B2B"
ACCENT, ACCENT_2, GOLD = "#0F6E56", "#C0502E", "#C9A227"   # teal = our estimate, rust = warning/threshold
SOURCE = ("Source: SUPPORT study (Connors et al., 1996), RHC data via Vanderbilt Biostatistics, "
          "n = 5,735 ICU patients, 1989 to 1994. Own analysis.")

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 11,
    "axes.titlesize": 14, "axes.labelsize": 11, "axes.labelcolor": DARK,
    "xtick.color": DARK, "ytick.color": DARK, "axes.edgecolor": GREY,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.major.size": 0, "ytick.major.size": 0,
})


def titles(ax, title, subtitle=None):
    """Action title (the takeaway, bold) plus a grey subtitle (what is shown)."""
    ax.set_title(title, loc="left", fontweight="bold", color=DARK, pad=24 if subtitle else 10)
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, color=GREY, fontsize=10, va="bottom")


def save(fig, name, source=SOURCE):
    """Add the source line, save as PNG, show inline."""
    fig.text(0.01, -0.02, source, fontsize=8, color=GREY, ha="left", va="top")
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    plt.show()


def fmt_ci(est, lo, hi, unit=" pp"):
    return f"{est:.1f}{unit} (95% CI {lo:.1f} to {hi:.1f})"


print("Setup done. Figures are saved to:", FIG_DIR.resolve())

Setup done. Figures are saved to: /home/user/causal-inference-heart-catheter-mortality/figures


---
## Step 1. Research question and data collection

### 1.1 Research question
- **Primary (ATE):** By how many percentage points does RHC in the first 24 hours change 30-day mortality, compared with managing the same patients without RHC?
- **Secondary (sensitivity):** The randomized trials found no effect on mortality. How strong would an unmeasured confounder, such as severity visible at the bedside but missing from the data, have to be to move our estimate to the randomized one?
- **Secondary (CATE):** Is the effect concentrated in some patients, for example by disease category or severity of illness?

Why it matters: Connors et al. (1996) reported higher mortality with RHC, an editorial called for a moratorium, and RHC use in the US fell by about two thirds over the following decade. Randomized trials later found no mortality effect. Whether the observational estimate reflects real harm or residual confounding is a question about the method as much as about the catheter.

### 1.2 Data collection
SUPPORT (Study to Understand Prognoses and Preferences for Outcomes and Risks of Treatments): seriously ill adults admitted to five US teaching hospitals between 1989 and 1994. The RHC subset of 5,735 patients from nine disease categories was analysed by Connors et al. and is published by Vanderbilt Biostatistics for methodological work on causal inference. It is an **observational** study: physicians decided who received a catheter, so treated and untreated patients may differ systematically (confounding by indication).

All covariates describe day 1, the day the patient qualified for the study and the day the treatment decision was made. Dates of discharge, death and last contact, as well as the 180-day death indicator, lie after the treatment and are not used as covariates.

In [2]:
# [CONCEPT] Download once, verify the checksum, then always read the local copy
# [WHY]     A checksum guarantees that every run, on every machine, analyses the identical file.
DATA_URL = "https://hbiostat.org/data/repo/rhc.csv"
DATA_SHA256 = "9ef4ab578be4b40ad5d97d3a7e08ffdc1f9f76aeeefee51b4996e4221556f8e8"


def load_rhc(path=DATA_DIR / "rhc.csv"):
    if not path.exists():
        path.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve(DATA_URL, path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != DATA_SHA256:
        raise ValueError(f"Checksum mismatch for {path} ({digest}). Delete the file and rerun to download it again.")
    return pd.read_csv(path, index_col=0)


df_raw = load_rhc()

W_COL, Y_COL = "swang1", "dth30"
# Variables measured after the treatment decision: never used as covariates
POST_COLS = ["dschdte", "dthdte", "lstctdte", "death", "t3d30"]
ID_COLS = ["ptid", "sadmdte"]

# Variable descriptions from the Vanderbilt data documentation (dth30 and t3d30 derived from the dates)
DESCRIPTIONS = {
    "swang1": "Right heart catheterization on day 1", "dth30": "Death within 30 days of study admission",
    "t3d30": "Days survived, capped at 30", "death": "Death at any time up to 180 days",
    "sadmdte": "Study admission date", "dschdte": "Hospital discharge date", "dthdte": "Date of death",
    "lstctdte": "Date of last contact", "ptid": "Patient ID",
    "age": "Age (years)", "sex": "Sex", "race": "Race", "edu": "Years of education", "income": "Income",
    "ninsclas": "Medical insurance", "cat1": "Primary disease category", "cat2": "Secondary disease category",
    "resp": "Admission diagnosis: respiratory", "card": "Admission diagnosis: cardiovascular",
    "neuro": "Admission diagnosis: neurological", "gastr": "Admission diagnosis: gastrointestinal",
    "renal": "Admission diagnosis: renal", "meta": "Admission diagnosis: metabolic",
    "hema": "Admission diagnosis: hematologic", "seps": "Admission diagnosis: sepsis",
    "trauma": "Admission diagnosis: trauma", "ortho": "Admission diagnosis: orthopedic",
    "adld3p": "Activities of daily living (ADL)", "das2d3pc": "Duke Activity Status Index (DASI)",
    "dnr1": "Do-not-resuscitate order on day 1", "ca": "Cancer",
    "surv2md1": "SUPPORT model estimate of 2-month survival", "aps1": "APACHE III acute physiology score",
    "scoma1": "Glasgow Coma Score", "wtkilo1": "Weight (kg)", "temp1": "Temperature",
    "meanbp1": "Mean blood pressure", "resp1": "Respiratory rate", "hrt1": "Heart rate",
    "pafi1": "PaO2/FiO2 ratio", "paco21": "PaCO2", "ph1": "pH", "wblc1": "White blood cell count",
    "hema1": "Hematocrit", "sod1": "Sodium", "pot1": "Potassium", "crea1": "Creatinine",
    "bili1": "Bilirubin", "alb1": "Albumin", "urin1": "Urine output",
    "cardiohx": "History: acute MI, PVD or severe cardiovascular symptoms",
    "chfhx": "History: congestive heart failure", "dementhx": "History: dementia, stroke or Parkinson's",
    "psychhx": "History: psychiatric illness or severe depression", "chrpulhx": "History: chronic pulmonary disease",
    "renalhx": "History: chronic renal disease or dialysis", "liverhx": "History: cirrhosis or hepatic failure",
    "gibledhx": "History: upper GI bleeding", "malighx": "History: solid tumor, metastasis or hematologic cancer",
    "immunhx": "History: immunosuppression, transplant, HIV or diabetes",
    "transhx": "Transfer (> 24 h) from another hospital", "amihx": "History: definite myocardial infarction",
}


def role(col):
    if col == W_COL:
        return "treatment W"
    if col == Y_COL:
        return "outcome Y"
    if col in POST_COLS:
        return "post-treatment (not used)"
    if col in ID_COLS:
        return "identifier (not used)"
    return "candidate confounder (day 1)"


data_dictionary = pd.DataFrame({
    "description": [DESCRIPTIONS[c] for c in df_raw.columns],
    "role": [role(c) for c in df_raw.columns],
    "missing_share": df_raw.isna().mean().round(3).to_numpy(),
}, index=df_raw.columns)

n_raw, n_rhc = len(df_raw), int((df_raw[W_COL] == "RHC").sum())
print(f"Full data: {n_raw:,} patients, {n_rhc:,} with RHC, {n_raw - n_rhc:,} without; "
      f"{df_raw[Y_COL].isna().sum()} missing outcomes")
print(f"Candidate confounders: {(data_dictionary['role'] == 'candidate confounder (day 1)').sum()}")
RESULTS.update({"n_raw": n_raw, "n_rhc_raw": n_rhc})
data_dictionary.sort_values(["role", "missing_share"], ascending=[True, False])

Full data: 5,735 patients, 2,184 with RHC, 3,551 without; 0 missing outcomes
Candidate confounders: 53


,description,role,missing_share
cat2,Secondary disease category,candidate confounder (day 1),0.791
adld3p,Activities of daily living (ADL),candidate confounder (day 1),0.749
urin1,Urine output,candidate confounder (day 1),0.528
cat1,Primary disease category,candidate confounder (day 1),0.000
ca,Cancer,candidate confounder (day 1),0.000
cardiohx,"History: acute MI, PVD or severe cardiovascula...",candidate confounder (day 1),0.000
chfhx,History: congestive heart failure,candidate confounder (day 1),0.000
dementhx,"History: dementia, stroke or Parkinson's",candidate confounder (day 1),0.000
psychhx,History: psychiatric illness or severe depression,candidate confounder (day 1),0.000
chrpulhx,History: chronic pulmonary disease,candidate confounder (day 1),0.000


**Checkpoint:** 5,735 patients (2,184 with RHC, 3,551 without), no missing outcomes, 53 candidate confounders.

---
## Next steps (in progress)
- **1.3 Missing covariates.** `cat2`, `adld3p` and `urin1` are missing for more than half of the patients. Decide per variable whether missingness carries information (for example, no secondary disease category) or whether the variable is dropped.
- **1.4 Analysis sample** and the unadjusted comparison of 30-day mortality.
- **Steps 2 to 8** as outlined in the pipeline table above.